In [16]:
# ============================================================
# TRAVELMATE AI
# CELL 1 — INSTALLATION
# ============================================================

!pip -q install "gradio==5.23.3" SpeechRecognition pydub

print("=" * 65)
print("✈️ TRAVELMATE AI")
print("Intelligent Travel Planning Assistant")
print("=" * 65)
print("✅ Gradio installed")
print("✅ SpeechRecognition installed")
print("✅ Pydub installed")
print("🚀 Environment ready")
print("=" * 65)

✈️ TRAVELMATE AI
Intelligent Travel Planning Assistant
✅ Gradio installed
✅ SpeechRecognition installed
✅ Pydub installed
🚀 Environment ready


In [17]:
# ============================================================
# TRAVELMATE AI
# CELL 2 — TRAVEL INTELLIGENCE
# ============================================================

import gradio as gr
import speech_recognition as sr

from pydub import AudioSegment

import tempfile
import os
import re
import random


# ============================================================
# DESTINATION DATABASE
# ============================================================

DESTINATIONS = {

    "Ooty": {
        "keywords": ["ooty", "udhagamandalam"],
        "theme": "Nature • Hills • Photography",
        "stay": "Hill-view hotel / cozy homestay",
        "food": [
            "Ooty homemade chocolates",
            "South Indian breakfast",
            "Nilgiri tea",
            "Local bakery snacks"
        ],
        "activities": [
            "Ooty Lake",
            "Botanical Garden",
            "Doddabetta Peak",
            "Tea Factory",
            "Rose Garden"
        ],
        "tips": [
            "Carry a light jacket because evenings can be cool.",
            "Start sightseeing early to avoid crowds.",
            "Try locally produced Nilgiri tea."
        ]
    },

    "Kodaikanal": {
        "keywords": ["kodaikanal", "kodai"],
        "theme": "Lake • Forest • Mountains",
        "stay": "Lake-view hotel / mountain cottage",
        "food": [
            "Homemade chocolates",
            "Hot coffee",
            "South Indian meals",
            "Local bakery items"
        ],
        "activities": [
            "Kodaikanal Lake",
            "Coaker's Walk",
            "Pillar Rocks",
            "Bryant Park",
            "Pine Forest"
        ],
        "tips": [
            "Carry comfortable shoes for walking.",
            "Keep a jacket for evening weather.",
            "Visit viewpoints early for better visibility."
        ]
    },

    "Goa": {
        "keywords": ["goa"],
        "theme": "Beaches • Cafés • Sunset",
        "stay": "Beach resort / boutique hotel",
        "food": [
            "Goan fish curry",
            "Pav bhaji",
            "Seafood",
            "Bebinca"
        ],
        "activities": [
            "Baga Beach",
            "Calangute Beach",
            "Fort Aguada",
            "Anjuna",
            "Sunset beach walk"
        ],
        "tips": [
            "Carry sunscreen and sunglasses.",
            "Keep comfortable beachwear.",
            "Plan beach visits around sunset."
        ]
    },

    "Munnar": {
        "keywords": ["munnar"],
        "theme": "Tea Gardens • Waterfalls • Nature",
        "stay": "Tea-estate resort / scenic homestay",
        "food": [
            "Kerala meals",
            "Appam and stew",
            "Puttu and kadala curry",
            "Local tea"
        ],
        "activities": [
            "Tea Gardens",
            "Mattupetty Dam",
            "Echo Point",
            "Tea Museum",
            "Waterfalls"
        ],
        "tips": [
            "Carry comfortable walking shoes.",
            "Try a tea plantation visit.",
            "Keep a light jacket for cooler areas."
        ]
    },

    "Manali": {
        "keywords": ["manali"],
        "theme": "Mountains • Adventure • Snow",
        "stay": "Mountain resort / wooden cottage",
        "food": [
            "Momos",
            "Thukpa",
            "North Indian meals",
            "Local Himachali dishes"
        ],
        "activities": [
            "Solang Valley",
            "Mall Road",
            "Hadimba Temple",
            "Old Manali",
            "Mountain viewpoints"
        ],
        "tips": [
            "Carry warm clothing.",
            "Keep comfortable trekking shoes.",
            "Check weather before adventure activities."
        ]
    },

    "Bengaluru": {
        "keywords": ["bengaluru", "bangalore"],
        "theme": "City • Technology • Food",
        "stay": "City hotel / serviced apartment",
        "food": [
            "Masala dosa",
            "Filter coffee",
            "Idli-vada",
            "Local cafés"
        ],
        "activities": [
            "Lalbagh Botanical Garden",
            "Cubbon Park",
            "Bangalore Palace",
            "Vidhana Soudha",
            "Local cafés"
        ],
        "tips": [
            "Plan around traffic-heavy hours.",
            "Use metro where convenient.",
            "Explore local cafés and food streets."
        ]
    },

    "Pondicherry": {
        "keywords": [
            "pondicherry",
            "puducherry",
            "pondy"
        ],
        "theme": "Beach • Cafés • Heritage",
        "stay": "Boutique hotel / heritage stay",
        "food": [
            "French-style pastries",
            "South Indian food",
            "Café dishes",
            "Fresh seafood"
        ],
        "activities": [
            "Promenade Beach",
            "White Town",
            "Auroville",
            "French Quarter",
            "Local cafés"
        ],
        "tips": [
            "Explore White Town early morning.",
            "Carry comfortable walking shoes.",
            "Try local cafés and French-inspired food."
        ]
    }
}


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):

    if not text:
        return ""

    text = text.lower().strip()

    return re.sub(r"\s+", " ", text)


# ============================================================
# DESTINATION DETECTION
# ============================================================

def detect_destination(text):

    text = clean_text(text)

    for destination, data in DESTINATIONS.items():

        for keyword in data["keywords"]:

            if keyword in text:
                return destination

    return "Ooty"


# ============================================================
# NUMBER OF DAYS
# ============================================================

def detect_days(text):

    text = clean_text(text)

    patterns = [
        r"(\d+)\s*day",
        r"(\d+)\s*days"
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            return max(1, min(int(match.group(1)), 10))

    return 3


# ============================================================
# BUDGET DETECTION
# ============================================================

def detect_budget(text):

    text = clean_text(text)

    patterns = [
        r"₹\s?(\d+)",
        r"rs\.?\s?(\d+)",
        r"rupees\s?(\d+)",
        r"budget\s*(?:of|is)?\s*₹?\s?(\d+)"
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:

            try:
                return int(match.group(1))
            except:
                pass

    return 10000


# ============================================================
# TRAVEL STYLE
# ============================================================

def detect_style(text):

    text = clean_text(text)

    styles = {
        "Adventure": [
            "adventure",
            "trek",
            "trekking",
            "adventurous"
        ],

        "Nature & Peace": [
            "nature",
            "peaceful",
            "relax",
            "quiet",
            "calm"
        ],

        "Photography": [
            "photo",
            "photography",
            "instagram",
            "scenic"
        ],

        "Family": [
            "family",
            "parents",
            "children"
        ],

        "Friends": [
            "friends",
            "friend"
        ],

        "Food & Culture": [
            "food",
            "foodie",
            "culture",
            "local food"
        ]
    }

    detected = []

    for style, keywords in styles.items():

        if any(word in text for word in keywords):
            detected.append(style)

    if not detected:
        return "Sightseeing"

    return " • ".join(detected[:2])


# ============================================================
# BUDGET BREAKDOWN
# ============================================================

def budget_breakdown(budget, days):

    transport = int(budget * 0.20)
    stay = int(budget * 0.30)
    food = int(budget * 0.20)
    activities = int(budget * 0.20)
    misc = budget - (
        transport +
        stay +
        food +
        activities
    )

    return {
        "Transport": transport,
        "Stay": stay,
        "Food": food,
        "Activities": activities,
        "Miscellaneous": misc
    }


# ============================================================
# PACKING LIST
# ============================================================

def packing_list(destination, style):

    items = [
        "Comfortable clothes",
        "Walking shoes",
        "Phone charger",
        "Power bank",
        "Travel documents",
        "Personal medicines",
        "Toiletries",
        "Water bottle"
    ]

    if destination in ["Ooty", "Kodaikanal", "Munnar", "Manali"]:
        items.append("Light jacket / warm clothes")

    if destination == "Goa":
        items.extend([
            "Sunscreen",
            "Sunglasses",
            "Beachwear"
        ])

    if "Adventure" in style:
        items.extend([
            "Small backpack",
            "Sports shoes"
        ])

    return list(dict.fromkeys(items))


# ============================================================
# ITINERARY
# ============================================================

def generate_itinerary(destination, days):

    data = DESTINATIONS[destination]

    activities = data["activities"]

    itinerary = []

    for day in range(1, days + 1):

        morning = activities[(day - 1) % len(activities)]

        afternoon = activities[day % len(activities)]

        evening = (
            "Explore local cafés and markets"
            if destination != "Manali"
            else "Enjoy an evening walk around Mall Road"
        )

        itinerary.append({
            "day": day,
            "morning": morning,
            "afternoon": afternoon,
            "evening": evening
        })

    return itinerary


# ============================================================
# MAIN TRAVEL ENGINE
# ============================================================

def create_travel_data(text):

    if not text or not text.strip():

        text = (
            "I want to visit Ooty for 3 days "
            "with a budget of ₹8000. "
            "I like nature and photography."
        )

    destination = detect_destination(text)

    days = detect_days(text)

    budget = detect_budget(text)

    style = detect_style(text)

    data = DESTINATIONS[destination]

    budget_data = budget_breakdown(
        budget,
        days
    )

    itinerary = generate_itinerary(
        destination,
        days
    )

    packing = packing_list(
        destination,
        style
    )

    return {
        "destination": destination,
        "days": days,
        "budget": budget,
        "style": style,
        "theme": data["theme"],
        "stay": data["stay"],
        "food": data["food"],
        "activities": data["activities"],
        "tips": data["tips"],
        "budget_data": budget_data,
        "itinerary": itinerary,
        "packing": packing
    }


print("=" * 65)
print("✈️ TRAVELMATE AI")
print("=" * 65)
print("✅ Destination database loaded")
print("✅ Travel planning engine loaded")
print("✅ Itinerary generator ready")
print("✅ Budget planner ready")
print("✅ Packing assistant ready")
print("=" * 65)
print("🚀 TravelMate AI intelligence is ready!")

✈️ TRAVELMATE AI
✅ Destination database loaded
✅ Travel planning engine loaded
✅ Itinerary generator ready
✅ Budget planner ready
✅ Packing assistant ready
🚀 TravelMate AI intelligence is ready!


In [ ]:
# ============================================================
# ✈️ TRAVELMATE AI
# CELL 3 — FINAL PROFESSIONAL UI
# ============================================================

import gradio as gr
import speech_recognition as sr
from pydub import AudioSegment
import tempfile
import os


# ============================================================
# REAL TRAVEL IMAGES
# ============================================================

IMAGES = {
    "Ooty":
        "https://images.unsplash.com/photo-1593693397690-362cb9666fc2?w=1200",

    "Kodaikanal":
        "https://images.unsplash.com/photo-1500534623283-312aade485b7?w=1200",

    "Goa":
        "https://images.unsplash.com/photo-1512343879784-a960bf40e7f2?w=1200",

    "Munnar":
        "https://images.unsplash.com/photo-1596422846543-75c6fc197f07?w=1200",

    "Manali":
        "https://images.unsplash.com/photo-1626621341517-bbf3d9990a23?w=1200",

    "Pondicherry":
        "https://images.unsplash.com/photo-1602216056096-3b40cc0c9944?w=1200"
}


# ============================================================
# RESULT FUNCTIONS
# ============================================================

def summary_html(data):

    return f"""
    <div class="summary-card">

        <div class="summary-title">
            ✨ YOUR PERSONALIZED TRIP
        </div>

        <div class="summary-grid">

            <div class="summary-item">
                <div class="summary-icon">📍</div>
                <div class="summary-label">DESTINATION</div>
                <div class="summary-value">
                    {data["destination"]}
                </div>
            </div>

            <div class="summary-item">
                <div class="summary-icon">📅</div>
                <div class="summary-label">DURATION</div>
                <div class="summary-value">
                    {data["days"]} Days
                </div>
            </div>

            <div class="summary-item">
                <div class="summary-icon">💰</div>
                <div class="summary-label">BUDGET</div>
                <div class="summary-value">
                    ₹{data["budget"]:,}
                </div>
            </div>

            <div class="summary-item">
                <div class="summary-icon">🎯</div>
                <div class="summary-label">TRAVEL STYLE</div>
                <div class="summary-value">
                    {data["style"]}
                </div>
            </div>

        </div>

    </div>
    """


def itinerary_html(data):

    cards = ""

    for item in data["itinerary"]:

        cards += f"""
        <div class="day-card">

            <div class="day-badge">
                DAY {item["day"]}
            </div>

            <h3>🌅 Morning</h3>
            <p>{item["morning"]}</p>

            <h3>☀️ Afternoon</h3>
            <p>{item["afternoon"]}</p>

            <h3>🌆 Evening</h3>
            <p>{item["evening"]}</p>

        </div>
        """

    return f"""
    <div class="result-card">

        <h2>🗓️ Day-by-Day Itinerary</h2>

        <div class="itinerary-grid">
            {cards}
        </div>

    </div>
    """


def budget_html(data):

    budget = data["budget_data"]

    return f"""
    <div class="result-card">

        <h2>💰 Smart Budget Plan</h2>

        <div class="budget-grid">

            <div class="budget-item">
                <span>🚗</span>
                <b>Transport</b>
                <strong>₹{budget["Transport"]:,}</strong>
            </div>

            <div class="budget-item">
                <span>🏨</span>
                <b>Stay</b>
                <strong>₹{budget["Stay"]:,}</strong>
            </div>

            <div class="budget-item">
                <span>🍴</span>
                <b>Food</b>
                <strong>₹{budget["Food"]:,}</strong>
            </div>

            <div class="budget-item">
                <span>🎟️</span>
                <b>Activities</b>
                <strong>₹{budget["Activities"]:,}</strong>
            </div>

            <div class="budget-item">
                <span>🛍️</span>
                <b>Miscellaneous</b>
                <strong>₹{budget["Miscellaneous"]:,}</strong>
            </div>

        </div>

        <div class="total-budget">

            <span>Total Trip Budget</span>

            <strong>
                ₹{data["budget"]:,}
            </strong>

        </div>

    </div>
    """


def stay_html(data):

    return f"""
    <div class="result-card">

        <h2>🏨 Stay Suggestions</h2>

        <div class="stay-box">

            <div class="stay-icon">🏨</div>

            <div>

                <h3>{data["stay"]}</h3>

                <p>
                    Recommended accommodation style for
                    <strong>{data["destination"]}</strong>.
                </p>

            </div>

        </div>

        <div class="check-list">

            <div>📍 Stay close to major attractions</div>

            <div>⭐ Prefer highly-rated accommodation</div>

            <div>🚗 Check transportation accessibility</div>

            <div>💰 Keep around 30% of the budget for stay</div>

        </div>

    </div>
    """


def food_html(data):

    foods = ""

    for food in data["food"]:

        foods += f"""
        <div class="food-item">
            🍴 {food}
        </div>
        """

    return f"""
    <div class="result-card">

        <h2>🍴 Food Recommendations</h2>

        <p>
            Try these local food options in
            <strong>{data["destination"]}</strong>.
        </p>

        <div class="food-grid">
            {foods}
        </div>

        <div class="theme-box">

            🌟 Travel Theme

            <strong>
                {data["theme"]}
            </strong>

        </div>

    </div>
    """


def packing_html(data):

    items = ""

    for item in data["packing"]:

        items += f"""
        <div class="packing-item">
            ☑️ {item}
        </div>
        """

    return f"""
    <div class="result-card">

        <h2>🎒 Smart Packing Checklist</h2>

        <div class="packing-grid">
            {items}
        </div>

        <div class="packing-tip">

            💡 Pack according to destination weather,
            trip duration and planned activities.

        </div>

    </div>
    """


def tips_html(data):

    tips = ""

    for tip in data["tips"]:

        tips += f"""
        <div class="tip-item">
            💡 {tip}
        </div>
        """

    return f"""
    <div class="result-card">

        <h2>💡 Travel Tips</h2>

        {tips}

        <div class="smart-tip">

            ✨ <strong>TravelMate Suggestion</strong>

            <br><br>

            Keep your itinerary flexible and leave
            some time for local exploration.

        </div>

    </div>
    """


# ============================================================
# COMPLETE TRAVEL PLAN
# ============================================================

def create_plan(text):

    data = create_travel_data(text)

    return (
        summary_html(data),
        itinerary_html(data),
        budget_html(data),
        stay_html(data),
        food_html(data),
        packing_html(data),
        tips_html(data)
    )


# ============================================================
# VOICE FUNCTIONS
# ============================================================

def convert_audio_to_wav(audio_path):

    if not audio_path:
        return None

    try:

        audio = AudioSegment.from_file(audio_path)

        wav_path = tempfile.mktemp(
            suffix=".wav"
        )

        audio.export(
            wav_path,
            format="wav"
        )

        return wav_path

    except Exception as e:

        print("Audio error:", e)

        return None


def speech_to_text(audio_path):

    if not audio_path:
        return "No voice recording received."

    wav_path = convert_audio_to_wav(
        audio_path
    )

    if not wav_path:
        return "Unable to process audio."

    recognizer = sr.Recognizer()

    try:

        with sr.AudioFile(wav_path) as source:

            audio = recognizer.record(source)

        text = recognizer.recognize_google(
            audio
        )

        return text

    except sr.UnknownValueError:

        return "Sorry, I could not understand the voice."

    except sr.RequestError:

        return "Speech recognition service is unavailable."

    except Exception as e:

        return f"Voice processing error: {e}"

    finally:

        try:

            if os.path.exists(wav_path):
                os.remove(wav_path)

        except:
            pass


def process_voice(audio_path):

    text = speech_to_text(audio_path)

    data = create_travel_data(text)

    return (
        text,
        summary_html(data),
        itinerary_html(data),
        budget_html(data),
        stay_html(data),
        food_html(data),
        packing_html(data),
        tips_html(data)
    )


# ============================================================
# EXAMPLE
# ============================================================

def load_example():

    example = (
        "I want to visit Ooty for 3 days "
        "with my friends with a budget of ₹8000. "
        "I like nature, photography, sightseeing "
        "and peaceful places."
    )

    return create_plan(example)


# ============================================================
# DESTINATION CARDS
# ============================================================

def destination_card(
    name,
    image,
    description,
    theme
):

    return f"""
    <div class="destination-card">

        <img
            src="{image}"
            class="destination-image"
        >

        <div class="destination-content">

            <h3>{name}</h3>

            <p>{description}</p>

            <span class="destination-theme">
                {theme}
            </span>

        </div>

    </div>
    """


# ============================================================
# EXPLORE SECTION
# ============================================================

explore_html = f"""

<div class="explore-grid">

{destination_card(
    "Ooty",
    IMAGES["Ooty"],
    "Beautiful hills, tea gardens and peaceful nature.",
    "🌿 Nature • Hills"
)}

{destination_card(
    "Kodaikanal",
    IMAGES["Kodaikanal"],
    "Lakes, forests, viewpoints and cool mountain weather.",
    "🏔️ Lake • Forest"
)}

{destination_card(
    "Goa",
    IMAGES["Goa"],
    "Beaches, cafés, sunsets and coastal experiences.",
    "🏖️ Beach • Sunset"
)}

{destination_card(
    "Munnar",
    IMAGES["Munnar"],
    "Tea plantations, waterfalls and scenic landscapes.",
    "🍃 Tea • Nature"
)}

{destination_card(
    "Manali",
    IMAGES["Manali"],
    "Mountains, adventure activities and beautiful valleys.",
    "🏔️ Mountain • Adventure"
)}

{destination_card(
    "Pondicherry",
    IMAGES["Pondicherry"],
    "Beaches, cafés, heritage streets and peaceful places.",
    "🌊 Beach • Heritage"
)}

</div>

"""


# ============================================================
# PROFESSIONAL CSS
# ============================================================

custom_css = """

/* ============================================================
   PAGE
============================================================ */

body {

    background:
        linear-gradient(
            135deg,
            #eef5ff,
            #f5f0ff,
            #eefcff
        ) !important;

}

.gradio-container {

    max-width: 1450px !important;

    margin: auto !important;

    padding: 24px !important;

    color: #172033 !important;

}


/* ============================================================
   HERO
============================================================ */

.hero {

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #6366f1,
            #7c3aed
        );

    border-radius: 28px;

    padding: 42px 25px;

    text-align: center;

    margin-bottom: 25px;

    box-shadow:
        0 18px 45px
        rgba(79,70,229,0.25);

}

.hero * {

    color: white !important;

}

.hero-icon {

    font-size: 50px;

}

.hero h1 {

    font-size: 44px;

    margin: 8px 0;

    font-weight: 800;

}

.hero p {

    font-size: 17px;

    margin: 5px;

}


/* ============================================================
   TABS
============================================================ */

.tab-nav {

    background: white !important;

    border-radius: 16px !important;

    padding: 7px !important;

    gap: 8px !important;

    border: 1px solid #dbe3ef !important;

}

.tab-nav button {

    background: #f1f5f9 !important;

    color: #172033 !important;

    border-radius: 12px !important;

    font-weight: 700 !important;

    padding: 14px 20px !important;

}

.tab-nav button.selected {

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #7c3aed
        ) !important;

    color: white !important;

}


/* ============================================================
   INPUT CARD
============================================================ */

.input-card {

    background: white !important;

    border: 1px solid #dbe3ef !important;

    border-radius: 20px !important;

    padding: 24px !important;

    box-shadow:
        0 10px 30px
        rgba(15,23,42,0.07) !important;

}


/* ============================================================
   TEXT
============================================================ */

h1,
h2,
h3,
p,
label {

    color: #172033 !important;

}

textarea,
input {

    background: #f8fafc !important;

    color: #172033 !important;

    border: 1px solid #cbd5e1 !important;

    border-radius: 13px !important;

}

textarea::placeholder,
input::placeholder {

    color: #64748b !important;

}


/* ============================================================
   BUTTON
============================================================ */

button.primary {

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #7c3aed
        ) !important;

    color: white !important;

    border: none !important;

    border-radius: 13px !important;

    font-weight: 700 !important;

    min-height: 48px !important;

}

button.primary * {

    color: white !important;

}


/* ============================================================
   SUMMARY
============================================================ */

.summary-card {

    background:
        linear-gradient(
            135deg,
            #dbeafe,
            #ede9fe
        );

    border-radius: 22px;

    padding: 24px;

    margin: 20px 0;

    border: 1px solid #bfdbfe;

}

.summary-title {

    color: #3730a3 !important;

    font-weight: 800;

    font-size: 18px;

    margin-bottom: 18px;

}

.summary-grid {

    display: grid;

    grid-template-columns:
        repeat(4,1fr);

    gap: 15px;

}

.summary-item {

    background: white;

    border-radius: 16px;

    padding: 18px;

    text-align: center;

    border: 1px solid #dbe3ef;

}

.summary-icon {

    font-size: 27px;

}

.summary-label {

    font-size: 11px;

    color: #64748b !important;

    font-weight: 800;

    margin-top: 6px;

}

.summary-value {

    color: #172033 !important;

    font-weight: 800;

    margin-top: 5px;

}


/* ============================================================
   RESULT CARD
============================================================ */

.result-card {

    background: white;

    border-radius: 20px;

    padding: 22px;

    border: 1px solid #dbe3ef;

    box-shadow:
        0 8px 25px
        rgba(15,23,42,0.06);

    margin-bottom: 15px;

}

.result-card h2 {

    color: #172033 !important;

    font-size: 21px;

}


/* ============================================================
   ITINERARY
============================================================ */

.itinerary-grid {

    display: grid;

    grid-template-columns:
        repeat(3,1fr);

    gap: 16px;

}

.day-card {

    background: #f8fbff;

    border: 1px solid #dbeafe;

    border-radius: 17px;

    padding: 18px;

}

.day-badge {

    display: inline-block;

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #6366f1
        );

    color: white !important;

    padding: 7px 12px;

    border-radius: 20px;

    font-size: 11px;

    font-weight: 800;

}

.day-card h3 {

    color: #334155 !important;

    font-size: 14px;

}

.day-card p {

    color: #64748b !important;

    font-size: 13px;

}


/* ============================================================
   BUDGET
============================================================ */

.budget-grid {

    display: grid;

    grid-template-columns:
        repeat(5,1fr);

    gap: 12px;

}

.budget-item {

    background: #f8fafc;

    border: 1px solid #e2e8f0;

    border-radius: 15px;

    padding: 15px;

    text-align: center;

}

.budget-item span {

    display: block;

    font-size: 25px;

}

.budget-item b {

    display: block;

    color: #334155 !important;

    font-size: 12px;

    margin: 5px;

}

.budget-item strong {

    color: #2563eb !important;

}

.total-budget {

    margin-top: 16px;

    padding: 17px;

    text-align: center;

    border-radius: 15px;

    background:
        linear-gradient(
            135deg,
            #2563eb,
            #7c3aed
        );

    color: white !important;

}

.total-budget span,
.total-budget strong {

    color: white !important;

}

.total-budget strong {

    display: block;

    font-size: 27px;

    margin-top: 5px;

}


/* ============================================================
   STAY
============================================================ */

.stay-box {

    display: flex;

    gap: 18px;

    align-items: center;

    background: #eff6ff;

    padding: 18px;

    border-radius: 16px;

}

.stay-icon {

    font-size: 40px;

}

.stay-box h3 {

    margin: 0;

    color: #172033 !important;

}

.stay-box p {

    color: #64748b !important;

}

.check-list {

    margin-top: 15px;

}

.check-list div {

    background: #f8fafc;

    padding: 11px;

    margin: 7px 0;

    border-radius: 10px;

    color: #334155 !important;

}


/* ============================================================
   FOOD
============================================================ */

.food-grid {

    display: grid;

    grid-template-columns:
        repeat(2,1fr);

    gap: 10px;

}

.food-item {

    background: #fff7ed;

    border: 1px solid #fed7aa;

    padding: 13px;

    border-radius: 12px;

    color: #7c2d12 !important;

}

.theme-box {

    background: #fef3c7;

    padding: 14px;

    border-radius: 12px;

    margin-top: 15px;

    color: #78350f !important;

}

.theme-box strong {

    color: #92400e !important;

}


/* ============================================================
   PACKING
============================================================ */

.packing-grid {

    display: grid;

    grid-template-columns:
        repeat(2,1fr);

    gap: 10px;

}

.packing-item {

    background: #ecfdf5;

    border: 1px solid #a7f3d0;

    padding: 12px;

    border-radius: 11px;

    color: #065f46 !important;

}

.packing-tip {

    margin-top: 15px;

    background: #eff6ff;

    padding: 14px;

    border-radius: 12px;

    color: #1e40af !important;

}


/* ============================================================
   TIPS
============================================================ */

.tip-item {

    background: #f8fafc;

    border-left: 4px solid #6366f1;

    padding: 13px;

    margin: 8px 0;

    border-radius: 8px;

    color: #334155 !important;

}

.smart-tip {

    margin-top: 16px;

    background: #f5f3ff;

    padding: 16px;

    border-radius: 13px;

    color: #5b21b6 !important;

}


/* ============================================================
   REAL IMAGE DESTINATION CARDS
============================================================ */

.explore-grid {

    display: grid;

    grid-template-columns:
        repeat(3,1fr);

    gap: 20px;

    margin-top: 20px;

}

.destination-card {

    background: white;

    border-radius: 20px;

    overflow: hidden;

    border: 1px solid #dbe3ef;

    box-shadow:
        0 10px 28px
        rgba(15,23,42,0.08);

    transition: 0.25s;

}

.destination-card:hover {

    transform: translateY(-5px);

    box-shadow:
        0 18px 35px
        rgba(37,99,235,0.16);

}

.destination-image {

    width: 100%;

    height: 210px;

    object-fit: cover;

    display: block;

}

.destination-content {

    padding: 18px;

}

.destination-content h3 {

    color: #172033 !important;

    font-size: 21px;

    margin: 0 0 7px 0;

}

.destination-content p {

    color: #64748b !important;

    font-size: 14px;

    min-height: 42px;

}

.destination-theme {

    display: inline-block;

    background: #eef2ff;

    color: #4338ca !important;

    padding: 7px 11px;

    border-radius: 20px;

    font-size: 12px;

    font-weight: 700;

}


/* ============================================================
   MOBILE
============================================================ */

@media(max-width:900px) {

    .summary-grid {

        grid-template-columns:
            repeat(2,1fr);

    }

    .itinerary-grid {

        grid-template-columns:
            repeat(2,1fr);

    }

    .budget-grid {

        grid-template-columns:
            repeat(2,1fr);

    }

    .explore-grid {

        grid-template-columns:
            repeat(2,1fr);

    }

}

@media(max-width:600px) {

    .gradio-container {

        padding: 10px !important;

    }

    .hero h1 {

        font-size: 30px;

    }

    .summary-grid,
    .itinerary-grid,
    .budget-grid,
    .food-grid,
    .packing-grid,
    .explore-grid {

        grid-template-columns: 1fr;

    }

    .tab-nav {

        flex-wrap: wrap !important;

    }

}

"""


# ============================================================
# BUILD APP
# ============================================================

with gr.Blocks(
    title="TravelMate AI",
    css=custom_css
) as app:


    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML("""

    <div class="hero">

        <div class="hero-icon">
            ✈️
        </div>

        <h1>
            TravelMate AI
        </h1>

        <p>
            Intelligent Travel Planning Assistant
        </p>

        <p>
            Plan Smarter • Travel Better • Explore More
        </p>

    </div>

    """)


    # ========================================================
    # TABS
    # ========================================================

    with gr.Tabs():


        # ====================================================
        # TEXT TRAVEL PLANNER
        # ====================================================

        with gr.TabItem(
            "📝 Text Travel Planner"
        ):

            gr.Markdown(
                """
# 🌍 Create Your Personalized Trip

Tell TravelMate where you want to go,
how many days you want to stay,
your budget and your travel interests.
"""
            )

            with gr.Row():

                with gr.Column(
                    scale=2,
                    elem_classes="input-card"
                ):

                    text_input = gr.Textbox(
                        label="✈️ Describe Your Trip",
                        placeholder=(
                            "Example: I want to visit Ooty "
                            "for 3 days with my friends "
                            "with a budget of ₹8000. "
                            "I like nature and photography."
                        ),
                        lines=6
                    )

                    with gr.Row():

                        generate_btn = gr.Button(
                            "✨ Generate Travel Plan",
                            variant="primary"
                        )

                        example_btn = gr.Button(
                            "📌 Load Example"
                        )


                with gr.Column(
                    scale=1,
                    elem_classes="input-card"
                ):

                    gr.Markdown(
                        """
## 💡 TravelMate AI

### What you get

🗓️ Personalized itinerary

💰 Budget planning

🏨 Stay suggestions

🍴 Food recommendations

🎒 Smart packing

💡 Travel tips
"""
                    )


            summary_output = gr.HTML()


            with gr.Row():

                itinerary_output = gr.HTML()

                budget_output = gr.HTML()


            with gr.Row():

                stay_output = gr.HTML()

                food_output = gr.HTML()


            with gr.Row():

                packing_output = gr.HTML()

                tips_output = gr.HTML()


            generate_btn.click(
                fn=create_plan,
                inputs=text_input,
                outputs=[
                    summary_output,
                    itinerary_output,
                    budget_output,
                    stay_output,
                    food_output,
                    packing_output,
                    tips_output
                ]
            )


            example_btn.click(
                fn=load_example,
                inputs=None,
                outputs=[
                    summary_output,
                    itinerary_output,
                    budget_output,
                    stay_output,
                    food_output,
                    packing_output,
                    tips_output
                ]
            )


        # ====================================================
        # VOICE TRAVEL PLANNER
        # ====================================================

        with gr.TabItem(
            "🎙️ Voice Travel Planner"
        ):

            gr.Markdown(
                """
# 🎙️ Plan Your Trip Using Your Voice

Speak naturally about your destination,
duration, budget and preferences.
"""
            )

            with gr.Row():

                with gr.Column(
                    elem_classes="input-card"
                ):

                    voice_input = gr.Audio(
                        sources=[
                            "microphone",
                            "upload"
                        ],
                        type="filepath",
                        label="🎙️ Record or Upload Your Voice"
                    )

                    voice_btn = gr.Button(
                        "🎤 Generate Voice Travel Plan",
                        variant="primary"
                    )


                with gr.Column(
                    elem_classes="input-card"
                ):

                    voice_text = gr.Textbox(
                        label="📝 Recognized Travel Request",
                        lines=7
                    )


            voice_summary = gr.HTML()


            with gr.Row():

                voice_itinerary = gr.HTML()

                voice_budget = gr.HTML()


            with gr.Row():

                voice_stay = gr.HTML()

                voice_food = gr.HTML()


            with gr.Row():

                voice_packing = gr.HTML()

                voice_tips = gr.HTML()


            voice_btn.click(
                fn=process_voice,
                inputs=voice_input,
                outputs=[
                    voice_text,
                    voice_summary,
                    voice_itinerary,
                    voice_budget,
                    voice_stay,
                    voice_food,
                    voice_packing,
                    voice_tips
                ]
            )


        # ====================================================
        # EXPLORE DESTINATIONS
        # ====================================================

        with gr.TabItem(
            "🌍 Explore Destinations"
        ):

            gr.Markdown(
                """
# 🌍 Explore Destinations

Discover beautiful destinations supported by TravelMate AI.
"""
            )

            gr.HTML(
                explore_html
            )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML("""

    <div style="
        text-align:center;
        padding:30px 10px;
        color:#475569;
    ">

        <div style="
            font-size:20px;
            font-weight:800;
            color:#3730a3;
        ">
            ✈️ TravelMate AI
        </div>

        <div style="
            margin-top:6px;
            color:#64748b;
        ">
            Intelligent Travel Planning Assistant
        </div>

        <div style="
            margin-top:12px;
            font-size:12px;
            color:#94a3b8;
        ">
            Built with Python • Gradio • Speech Recognition
        </div>

    </div>

    """)


# ============================================================
# LAUNCH
# ============================================================

print("=" * 65)
print("✈️ TRAVELMATE AI")
print("=" * 65)
print("🚀 Starting professional interface...")
print("📸 Real destination images enabled")
print("🎙️ Voice planner enabled")
print("📝 Text planner enabled")
print("🌍 Destination explorer enabled")
print("=" * 65)

app.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_2199/355054028.py:1432: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✈️ TRAVELMATE AI
🚀 Starting professional interface...
📸 Real destination images enabled
🎙️ Voice planner enabled
📝 Text planner enabled
🌍 Destination explorer enabled
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://efce9f216b6e7a5d6d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
